# MES Mean Reversion / Momentum Transition Analysis

**Phase 2, Task 2** | MES Micro E-mini S&P 500 Futures

This notebook answers: *At what timescale does MES transition from mean reversion to momentum?*

---

**Analyses:**
1. Autocorrelation profile across lags (in minutes)
2. Hurst exponent by resampled window size (30s–60m)
3. AR(1) mean reversion coefficient by timescale — find the crossing point
4. Time-of-day segmentation (open / midday / close)

**Expected finding:** Mean reversion peaks at 4–8 min on MES, transitioning to momentum at 15–30 min.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

from src.analysis.regime_characterization import (
    run_full_analysis,
    segment_by_time_of_day,
    compute_acf_profile,
    compute_hurst_by_window,
    compute_ar1_by_timescale,
    find_crossing_point,
    print_regime_report,
    DEFAULT_WINDOWS,
)
from src.analysis.bar_statistics import compute_log_returns

sns.set_theme(style="whitegrid")

# ── Load 1s bars (adjust years to your data) ──
PARQUET_DIR = "../data/parquet"
START_YEAR = 2024
END_YEAR = 2024

frames = []
for year in range(START_YEAR, END_YEAR + 1):
    path = os.path.join(PARQUET_DIR, f"year={year}", "data.parquet")
    if os.path.exists(path):
        df = pl.read_parquet(path)
        frames.append(df)
        print(f"Loaded {path} ({len(df):,} rows)")

df_1s = pl.concat(frames).sort("timestamp")
print(f"\nTotal: {len(df_1s):,} 1-second bars")
print(f"Date range: {df_1s['timestamp'].min()} to {df_1s['timestamp'].max()}")

## 1. Autocorrelation Profile

ACF of 1s log returns across lags, converted to approximate minutes. Green shading = mean reversion zone (negative ACF).

In [ ]:
returns_1s = compute_log_returns(df_1s["close"].to_numpy())
acf_result = compute_acf_profile(returns_1s, max_lag=300)

lags_minutes = np.array(acf_result.lags) / 60.0
acf_vals = np.array(acf_result.acf_values)
ci_lower = np.array(acf_result.confidence_lower)
ci_upper = np.array(acf_result.confidence_upper)

fig, ax = plt.subplots(figsize=(12, 5))

# Shade mean-reversion zone (negative ACF)
ax.fill_between(lags_minutes, acf_vals, 0,
                where=(acf_vals < 0), alpha=0.3, color="green", label="Mean reversion zone")
ax.fill_between(lags_minutes, acf_vals, 0,
                where=(acf_vals > 0), alpha=0.3, color="steelblue", label="Momentum zone")

ax.plot(lags_minutes, acf_vals, color="black", linewidth=1, label="ACF")
ax.fill_between(lags_minutes, ci_lower, ci_upper, alpha=0.15, color="gray", label="95% CI")
ax.axhline(y=0, color="black", linewidth=0.5)

ax.set_xlabel("Lag (minutes)")
ax.set_ylabel("Autocorrelation")
ax.set_title("MES 1s Return Autocorrelation Profile")
ax.legend(loc="upper right")
ax.set_xlim(0, 5)  # Focus on first 5 minutes where structure is visible

plt.tight_layout()
plt.savefig("../docs/phase2/acf_profile.png", dpi=150, bbox_inches="tight")
plt.show()

# Summary stats
n_sig = len(acf_result.significant_lags)
print(f"\nSignificant lags: {n_sig}/{len(acf_result.lags)}")
if acf_result.significant_lags:
    first_sig = acf_result.significant_lags[:10]
    print(f"First significant lags (seconds): {first_sig}")
    print(f"First significant lags (minutes): {[f'{l/60:.1f}' for l in first_sig]}")

## 2. Hurst Exponent by Window Size

H < 0.5 = mean-reverting, H = 0.5 = random walk, H > 0.5 = trending.

In [ ]:
hurst_result = compute_hurst_by_window(df_1s)

fig, ax = plt.subplots(figsize=(10, 5))

valid_mask = [not np.isnan(h) for h in hurst_result.hurst_values]
labels = [l for l, v in zip(hurst_result.window_labels, valid_mask) if v]
values = [h for h, v in zip(hurst_result.hurst_values, valid_mask) if v]
colors = ["green" if h < 0.45 else "gold" if h <= 0.55 else "steelblue" for h in values]

bars = ax.bar(labels, values, color=colors, edgecolor="black", linewidth=0.5)
ax.axhline(y=0.5, color="red", linestyle="--", linewidth=1.5, label="H = 0.5 (random walk)")

# Annotate values
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{val:.3f}", ha="center", va="bottom", fontsize=10)

ax.set_xlabel("Window Size")
ax.set_ylabel("Hurst Exponent (H)")
ax.set_title("Hurst Exponent by Resampled Window Size")
ax.legend()
ax.set_ylim(0, max(values) + 0.15 if values else 1.0)

# Add zone labels
ax.text(0.02, 0.35, "Mean Reversion Zone", transform=ax.transAxes,
        fontsize=10, color="green", fontstyle="italic")
ax.text(0.02, 0.7, "Trending Zone", transform=ax.transAxes,
        fontsize=10, color="steelblue", fontstyle="italic")

plt.tight_layout()
plt.savefig("../docs/phase2/hurst_by_window.png", dpi=150, bbox_inches="tight")
plt.show()

# Print table
print(f"\n{'Window':<8} {'Hurst':>7} {'Classification':<16}")
print("-" * 35)
for lbl, h, cls in zip(hurst_result.window_labels, hurst_result.hurst_values, hurst_result.classifications):
    h_str = f"{h:.3f}" if not np.isnan(h) else "N/A"
    print(f"{lbl:<8} {h_str:>7} {cls:<16}")

## 3. AR(1) Mean Reversion Coefficient by Timescale

Positive = mean reverting (returns tend to reverse). Negative = momentum (returns persist).
The crossing point is the maximum hold time for mean reversion strategies.

In [ ]:
ar1_results = compute_ar1_by_timescale(df_1s)
crossing = find_crossing_point(ar1_results)

ts_labels = [r.timescale_label for r in ar1_results]
ts_seconds = [r.timescale_seconds for r in ar1_results]
mr_coeffs = [r.mean_reversion_coeff for r in ar1_results]
p_values = [r.p_value for r in ar1_results]

fig, ax = plt.subplots(figsize=(10, 5))

colors = ["green" if c > 0 else "steelblue" for c in mr_coeffs]
ax.bar(ts_labels, mr_coeffs, color=colors, edgecolor="black", linewidth=0.5)
ax.axhline(y=0, color="red", linestyle="--", linewidth=1.5)

# Annotate with coefficient values
for i, (label, coeff, p) in enumerate(zip(ts_labels, mr_coeffs, p_values)):
    if not np.isnan(coeff):
        sig = "*" if p < 0.05 else ""
        ax.text(i, coeff + (0.005 if coeff >= 0 else -0.01),
                f"{coeff:+.4f}{sig}", ha="center", va="bottom" if coeff >= 0 else "top", fontsize=9)

# Mark crossing point
if crossing:
    ax.axvline(x=crossing.timescale_label if crossing.timescale_label in ts_labels else 2.5,
               color="orange", linestyle=":", linewidth=2)
    ax.text(0.5, 0.95, f"Crossing point: {crossing.timescale_label}",
            transform=ax.transAxes, fontsize=11, color="orange",
            ha="center", va="top", fontweight="bold",
            bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))

ax.set_xlabel("Timescale")
ax.set_ylabel("Mean Reversion Coefficient")
ax.set_title("AR(1) Mean Reversion Coefficient by Timescale\n(+ = reversion, - = momentum, * = p < 0.05)")

plt.tight_layout()
plt.savefig("../docs/phase2/ar1_by_timescale.png", dpi=150, bbox_inches="tight")
plt.show()

# Print table
print(f"\n{'Timescale':<10} {'MR Coeff':>10} {'p-value':>10} {'Significant':>12}")
print("-" * 45)
for r in ar1_results:
    if not np.isnan(r.mean_reversion_coeff):
        sig = "YES" if r.p_value < 0.05 else "no"
        print(f"{r.timescale_label:<10} {r.mean_reversion_coeff:>+10.4f} {r.p_value:>10.4f} {sig:>12}")

if crossing:
    print(f"\nCROSSING POINT: {crossing.timescale_label} ({crossing.timescale_seconds:.0f}s)")
    print(f"  Below: {crossing.below_regime}")
    print(f"  Above: {crossing.above_regime}")
else:
    print("\nNo crossing point detected.")

## 4. Time-of-Day Segmentation

Autocorrelation profile for three RTH segments:
- **Open** (9:30–11:00 AM): Typically mixed momentum/reversion
- **Midday** (11:00 AM–2:00 PM): Dead zone, expected stronger mean reversion
- **Close** (2:00–4:00 PM): Renewed directional activity

In [ ]:
segments = segment_by_time_of_day(df_1s)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
segment_colors = {"open": "tab:orange", "midday": "tab:green", "close": "tab:blue"}
segment_titles = {"open": "Open (9:30-11:00)", "midday": "Midday (11:00-14:00)", "close": "Close (14:00-16:00)"}

tod_reports = []

for ax, (name, seg_df) in zip(axes, segments.items()):
    if len(seg_df) < 200:
        ax.set_title(f"{segment_titles[name]}\n(insufficient data)")
        continue

    seg_returns = compute_log_returns(seg_df["close"].to_numpy())
    seg_acf = compute_acf_profile(seg_returns, max_lag=300)
    lags_min = np.array(seg_acf.lags) / 60.0
    acf_vals = np.array(seg_acf.acf_values)
    ci_lo = np.array(seg_acf.confidence_lower)
    ci_hi = np.array(seg_acf.confidence_upper)

    ax.fill_between(lags_min, acf_vals, 0,
                    where=(acf_vals < 0), alpha=0.3, color="green")
    ax.fill_between(lags_min, acf_vals, 0,
                    where=(acf_vals > 0), alpha=0.3, color="steelblue")
    ax.plot(lags_min, acf_vals, color=segment_colors[name], linewidth=1)
    ax.fill_between(lags_min, ci_lo, ci_hi, alpha=0.1, color="gray")
    ax.axhline(y=0, color="black", linewidth=0.5)
    ax.set_title(f"{segment_titles[name]}\n({len(seg_df):,} bars)")
    ax.set_xlabel("Lag (minutes)")
    ax.set_xlim(0, 5)

    # Also run full analysis for the findings table
    if len(seg_df) > 3600:  # need at least 1 hour for meaningful analysis
        report = run_full_analysis(seg_df, label=name)
        tod_reports.append(report)

axes[0].set_ylabel("Autocorrelation")
fig.suptitle("ACF by Time of Day", fontsize=14, y=1.02)

plt.tight_layout()
plt.savefig("../docs/phase2/acf_by_tod.png", dpi=150, bbox_inches="tight")
plt.show()

# Print per-segment crossing points
for r in tod_reports:
    cp = r.crossing_point
    if cp:
        print(f"{r.label}: crossing at {cp.timescale_label}")
    else:
        print(f"{r.label}: no crossing point detected")

## 5. Findings Table

Summary of regime characteristics across timescales.

In [ ]:
# Run full-session analysis
full_report = run_full_analysis(df_1s, label="full-session")

print("=" * 75)
print("FINDINGS: MES REGIME CHARACTERISTICS BY TIMESCALE")
print("=" * 75)
print(f"\n{'Timescale':<10} {'Hurst':>7} {'MR Coeff':>10} {'Dominant Effect':<18} {'Tradable?':<12}")
print("-" * 60)

hw = full_report.hurst_by_window
ar1_map = {r.timescale_label: r for r in full_report.ar1_results}

for lbl, h, cls in zip(hw.window_labels, hw.hurst_values, hw.classifications):
    ar1 = ar1_map.get(lbl)
    mr_str = f"{ar1.mean_reversion_coeff:+.4f}" if ar1 and not np.isnan(ar1.mean_reversion_coeff) else "N/A"
    h_str = f"{h:.3f}" if not np.isnan(h) else "N/A"

    if ar1 and not np.isnan(ar1.mean_reversion_coeff):
        if ar1.mean_reversion_coeff > 0.01:
            effect = "Mean Reversion"
        elif ar1.mean_reversion_coeff < -0.01:
            effect = "Momentum"
        else:
            effect = "Random Walk"
    else:
        effect = cls.replace("-", " ").title()

    # Tradability assessment
    if lbl in ("1s", "30s"):
        tradable = "No (spread)"
    elif effect == "Mean Reversion" and ar1 and ar1.p_value < 0.05:
        tradable = "YES"
    elif effect == "Momentum" and ar1 and ar1.p_value < 0.05:
        tradable = "YES"
    else:
        tradable = "Weak"

    print(f"{lbl:<10} {h_str:>7} {mr_str:>10} {effect:<18} {tradable:<12}")

if full_report.crossing_point:
    cp = full_report.crossing_point
    print(f"\nCROSSING POINT: {cp.timescale_label} ({cp.timescale_seconds:.0f} seconds)")
    print(f"  Below {cp.timescale_label}: {cp.below_regime} dominant")
    print(f"  Above {cp.timescale_label}: {cp.above_regime} dominant")
else:
    print("\nNo clean crossing point detected — check individual timescales above.")

print("\n" + "=" * 75)

## 6. Strategy Implications

Interpret the findings for Phase 3 strategy design.

In [ ]:
print("STRATEGY IMPLICATIONS")
print("=" * 60)

cp = full_report.crossing_point

if cp:
    cp_min = cp.timescale_seconds / 60
    print(f"""
Based on the regime analysis of MES 1-second bars:

1. VWAP REVERSION STRATEGY
   - Mean reversion dominates below {cp.timescale_label}
   - Target exits within {cp_min:.0f} minutes of entry
   - Beyond this window, momentum takes over and stops get hit

2. ORB (OPENING RANGE BREAKOUT) STRATEGY
   - Momentum holds can extend beyond {cp.timescale_label}
   - Use trailing stops after {cp_min:.0f} minutes to ride momentum

3. GENERAL PARAMETER GUIDANCE
   - Max hold time for reversion trades: {cp_min:.0f} minutes
   - Momentum confirmation window: > {cp_min:.0f} minutes
   - Avoid sub-30s signals (bid-ask bounce, not tradable edge)
""")
else:
    print("""
No clean crossing point was detected. Review the AR(1) coefficients
and Hurst exponents above to identify the dominant regime at each
timescale. The per-timescale findings should still inform strategy
parameter selection for Phase 3.
""")

# Time-of-day implications
print("4. TIME-OF-DAY GUIDANCE")
for r in tod_reports:
    cp_tod = r.crossing_point
    if cp_tod:
        print(f"   {r.label.upper()}: crossing at {cp_tod.timescale_label} — "
              f"reversion below, momentum above")
    else:
        # Check dominant regime from AR1 results
        valid_ar1 = [a for a in r.ar1_results if not np.isnan(a.mean_reversion_coeff)]
        if valid_ar1:
            avg_mr = np.mean([a.mean_reversion_coeff for a in valid_ar1])
            regime = "mean-reverting" if avg_mr > 0 else "momentum-leaning"
            print(f"   {r.label.upper()}: no crossing detected, overall {regime}")

print("\n" + "=" * 60)
print("Save these findings — they are the parameter bible for Phase 3 strategies.")